# DE2 — Final Project Notebook
Author: JABRY — ESIEE 2025-2026 — Data Engineering II
Track: A — Esports (OpenDota-style)
Path chosen for iterative workload: A — Graph Processing (PageRank)

Pipeline: ETL bronze→silver→gold • Streaming • Text → Inverted Index • Iterative PageRank • LLM data readiness • Evidence

## 0. Load config & start Spark

In [1]:
import os, yaml, pathlib, datetime, time, json, csv, io, contextlib, shutil
from urllib.parse import urlparse
from datetime import datetime
from pyspark.sql import SparkSession, functions as F, types as T

with open("de2_project_config.yml") as f:
    CFG = yaml.safe_load(f)

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)

spark = (SparkSession.builder
    .appName("de2-project")
    .master("local[*]")
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST)
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS)
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS)
    .config("spark.sql.shuffle.partitions", str(CFG["layout"]["shuffle_partitions"]))
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate())
spark.sparkContext.setCheckpointDir("checkpoint_tmp")

ui = spark.sparkContext.uiWebUrl
print("Spark:", spark.version)
print("Spark UI:", ui)
if ui:
    print("Spark UI (WSL/Windows browser):", f"http://localhost:{urlparse(ui).port or 4040}")

pathlib.Path(CFG["paths"]["proof"]).mkdir(parents=True, exist_ok=True)
pathlib.Path("outputs/project").mkdir(parents=True, exist_ok=True)

# Metrics log we'll append to throughout
METRICS = []
def log_metric(stage, task, name, value, notes=""):
    METRICS.append({
        "run_id": len(METRICS) + 1,
        "stage": stage,
        "task": task,
        "metric_name": name,
        "metric_value": value,
        "notes": notes,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    })

def save_plan(df, path):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain("formatted")
    with open(path, "w") as f:
        f.write(buf.getvalue())
    print("Plan saved:", path)

def dir_size(path):
    p = pathlib.Path(path)
    if not p.exists():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

print("Config loaded. Track:", CFG["track"])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 22:44:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040
Config loaded. Track: A_Esports


## 1. Bronze — landing raw data
Raw CSV shards land here, untouched, immutable. Schema is inferred for inspection but we capture the row count and on-disk footprint for evidence.

In [2]:
raw_glob = CFG["paths"]["raw_csv_glob"]
bronze = CFG["paths"]["bronze"]

t0 = time.time()
df_raw = (spark.read.option("header", "true")
          .csv(raw_glob))
n_raw = df_raw.count()
print(f"Raw rows ingested: {n_raw:,}")
df_raw.printSchema()
df_raw.show(3, truncate=80)

if pathlib.Path(bronze).exists():
    shutil.rmtree(bronze)
df_raw.write.mode("overwrite").csv(bronze)
bronze_bytes = dir_size(bronze)
elapsed_ms = (time.time() - t0) * 1000

print(f"Bronze written to {bronze}  •  {bronze_bytes:,} bytes  •  {elapsed_ms:.0f} ms")
log_metric("etl", "bronze_land", "rows", n_raw, "raw CSV landed immutably")
log_metric("etl", "bronze_land", "bytes", bronze_bytes, "on-disk footprint")
log_metric("etl", "bronze_land", "elapsed_ms", round(elapsed_ms, 1), "")

26/05/16 22:55:41 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/matches_*.csv.
java.io.FileNotFoundException: File data/raw/matches_*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Resolve

Raw rows ingested: 30,000
root
 |-- match_id: string (nullable = true)
 |-- match_end_time: string (nullable = true)
 |-- team: string (nullable = true)
 |-- hero: string (nullable = true)
 |-- role: string (nullable = true)
 |-- kills: string (nullable = true)
 |-- deaths: string (nullable = true)
 |-- duration_sec: string (nullable = true)
 |-- region: string (nullable = true)
 |-- patch_version: string (nullable = true)
 |-- commentary_text: string (nullable = true)

+--------+-------------------+------+-------+-------+-----+------+------------+------+-------------+--------------------------------------------------------------------------------+
|match_id|     match_end_time|  team|   hero|   role|kills|deaths|duration_sec|region|patch_version|                                                                 commentary_text|
+--------+-------------------+------+-------+-------+-----+------+------------+------+-------------+-------------------------------------------------------------

## 2. Silver — cleaning, typing, schema contracts, dedup
Apply explicit types, drop nulls in critical columns, deduplicate by `(match_id, hero)` natural key. Save as Parquet.

In [3]:
silver = CFG["paths"]["silver"]

t0 = time.time()
df_silver = (df_raw
    .withColumn("match_id",       F.col("match_id").cast(T.IntegerType()))
    .withColumn("match_end_time", F.to_timestamp("match_end_time"))
    .withColumn("team",           F.col("team").cast(T.StringType()))
    .withColumn("hero",           F.col("hero").cast(T.StringType()))
    .withColumn("role",           F.col("role").cast(T.StringType()))
    .withColumn("kills",          F.col("kills").cast(T.IntegerType()))
    .withColumn("deaths",         F.col("deaths").cast(T.IntegerType()))
    .withColumn("duration_sec",   F.col("duration_sec").cast(T.IntegerType()))
    .withColumn("region",         F.col("region").cast(T.StringType()))
    .withColumn("patch_version",  F.col("patch_version").cast(T.StringType()))
    .withColumn("commentary_text", F.col("commentary_text").cast(T.StringType()))
    # contract: critical columns must not be null
    .dropna(subset=["match_id", "match_end_time", "team", "hero"])
    .dropDuplicates(["match_id", "hero"]))

if pathlib.Path(silver).exists():
    shutil.rmtree(silver)
df_silver.write.mode("overwrite").parquet(silver)
n_silver = spark.read.parquet(silver).count()
silver_bytes = dir_size(silver)
elapsed_ms = (time.time() - t0) * 1000

save_plan(df_silver, f"{CFG['paths']['proof']}/plan_etl.txt")
print(f"Silver rows: {n_silver:,}  •  {silver_bytes:,} bytes  •  {elapsed_ms:.0f} ms")
print(f"Dropped {n_raw - n_silver:,} rows (nulls + duplicates)")
log_metric("etl", "silver_clean", "rows", n_silver, "after type cast + null drop + dedup")
log_metric("etl", "silver_clean", "bytes", silver_bytes, "Parquet output")
log_metric("etl", "silver_clean", "elapsed_ms", round(elapsed_ms, 1), "")
log_metric("etl", "silver_clean", "rows_dropped", n_raw - n_silver, "contract violations + duplicates")

Plan saved: proof/plan_etl.txt
Silver rows: 30,000  •  1,598,682 bytes  •  3156 ms
Dropped 0 rows (nulls + duplicates)


## 3. Gold — analytics tables partitioned by region

In [4]:
gold = CFG["paths"]["gold"]
partition_by = CFG["layout"]["partition_by"]

t0 = time.time()
df_silver_in = spark.read.parquet(silver)

# Gold table 1: team performance by region + patch
gold_team = (df_silver_in.groupBy("region", "patch_version", "team")
    .agg(F.count("*").alias("picks"),
         F.avg("kills").alias("avg_kills"),
         F.avg("deaths").alias("avg_deaths"),
         F.avg("duration_sec").alias("avg_duration_sec")))

# Gold table 2: hero pick stats by region
gold_hero = (df_silver_in.groupBy("region", "hero")
    .agg(F.count("*").alias("picks"),
         F.avg("kills").alias("avg_kills")))

for path, df in [("q1_team_perf", gold_team), ("q2_hero_picks", gold_hero)]:
    out = f"{gold}/{path}"
    if pathlib.Path(out).exists():
        shutil.rmtree(out)
    df.write.mode("overwrite").partitionBy(*partition_by).parquet(out)

gold_bytes = dir_size(gold)
elapsed_ms = (time.time() - t0) * 1000
n_gold_team = spark.read.parquet(f"{gold}/q1_team_perf").count()
n_gold_hero = spark.read.parquet(f"{gold}/q2_hero_picks").count()

print(f"Gold tables written  •  team_perf={n_gold_team:,} rows  •  hero_picks={n_gold_hero:,} rows")
print(f"Gold footprint: {gold_bytes:,} bytes  •  {elapsed_ms:.0f} ms")
gold_team.show(5)
gold_hero.orderBy(F.desc("picks")).show(5)

log_metric("etl", "gold_build", "team_perf_rows", n_gold_team, f"partitioned by {partition_by}")
log_metric("etl", "gold_build", "hero_picks_rows", n_gold_hero, "")
log_metric("etl", "gold_build", "bytes", gold_bytes, "total gold footprint")
log_metric("etl", "gold_build", "elapsed_ms", round(elapsed_ms, 1), "")

Gold tables written  •  team_perf=360 rows  •  hero_picks=250 rows
Gold footprint: 138,272 bytes  •  2309 ms
+------+-------------+----+-----+-----------------+------------------+------------------+
|region|patch_version|team|picks|        avg_kills|        avg_deaths|  avg_duration_sec|
+------+-------------+----+-----+-----------------+------------------+------------------+
|    CN|        7.35a|  EG|   80|          17.7125|           16.1375| 2211.551282051282|
|    CN|        7.34a| FNC|   90|16.11111111111111|17.144444444444446|2330.0581395348836|
|    KR|        7.35a|  T2|   80|             16.7|            19.625|         2201.7625|
|    EU|         7.34|  G2|   60|            18.05|16.466666666666665|2291.1833333333334|
|    CN|        7.34a|  EG|   35|             15.2|18.542857142857144| 2333.323529411765|
+------+-------------+----+-----+-----------------+------------------+------------------+
only showing top 5 rows
+------+-------+-----+------------------+
|region|   hero

## 4. Streaming Pipeline — file source → window + watermark → Parquet sink

In [5]:
stream_schema = T.StructType([
    T.StructField("match_id",       T.IntegerType(),   True),
    T.StructField("match_end_time", T.TimestampType(), True),
    T.StructField("team",           T.StringType(),    True),
    T.StructField("kills",          T.IntegerType(),   True),
    T.StructField("deaths",         T.IntegerType(),   True),
    T.StructField("duration_sec",   T.IntegerType(),   True),
])

landing = CFG["paths"]["streaming_landing"]
source = CFG["paths"]["streaming_source"]
pathlib.Path(landing).mkdir(parents=True, exist_ok=True)
for f in pathlib.Path(landing).glob("*.json"):
    f.unlink()

df_stream = (spark.readStream
    .schema(stream_schema)
    .option("maxFilesPerTrigger", CFG["streaming"]["max_files_per_trigger"])
    .json(landing))

windowed = (df_stream
    .withWatermark("match_end_time", CFG["streaming"]["watermark"])
    .groupBy(F.window("match_end_time", CFG["streaming"]["window_duration"]),
             F.col("team"))
    .agg(F.count("*").alias("event_count"),
         F.avg("kills").alias("avg_kills"),
         F.avg("duration_sec").alias("avg_duration_sec")))

save_plan(windowed, f"{CFG['paths']['proof']}/plan_streaming.txt")

sink_path = "outputs/project/streaming"
ck_path = "outputs/project/streaming_checkpoint"
for p in [sink_path, ck_path]:
    if pathlib.Path(p).exists():
        shutil.rmtree(p)

query = (windowed.writeStream.format("parquet")
    .outputMode("append")
    .option("path", sink_path)
    .option("checkpointLocation", ck_path)
    .trigger(processingTime=CFG["streaming"]["trigger_interval"])
    .start())
print("Streaming query started:", query.id, "active:", query.isActive)

Plan saved: proof/plan_streaming.txt
Streaming query started: ac41c487-a8a4-4ee1-a8fc-7f34a8f91daf active: True


In [6]:
# Feed the streaming source one file at a time
source_files = sorted(pathlib.Path(source).glob("*.json"))
print(f"Feeding {len(source_files)} files into {landing}...")
for i, src in enumerate(source_files, start=1):
    shutil.copy(src, pathlib.Path(landing) / src.name)
    print(f"  [{i}/{len(source_files)}] dropped {src.name}")
    time.sleep(12)
time.sleep(15)

progress = query.lastProgress
print("\nLast progress:")
print(json.dumps(progress, indent=2, default=str))

with open(f"{CFG['paths']['proof']}/query_progress.json", "w") as f:
    json.dump(progress, f, indent=2, default=str)

query.stop()
print("\nStreaming query stopped.")

# Metrics
if progress:
    log_metric("streaming", "file_source_pipeline", "numInputRows", progress.get("numInputRows", 0), "last batch")
    log_metric("streaming", "file_source_pipeline", "processedRowsPerSecond", round(float(progress.get("processedRowsPerSecond") or 0), 2), "")
    log_metric("streaming", "file_source_pipeline", "triggerExecution_ms", (progress.get("durationMs") or {}).get("triggerExecution", 0), "")
    state = (progress.get("stateOperators") or [{}])[0]
    log_metric("streaming", "file_source_pipeline", "stateRowsTotal", state.get("numRowsTotal", 0), "")

# Verify sink
n_sink = spark.read.parquet(sink_path).count()
print(f"Streaming Parquet sink: {n_sink} windowed rows")
log_metric("streaming", "file_source_pipeline", "sink_rows", n_sink, "windows × teams written")

Feeding 10 files into data/streaming/landing/...
  [1/10] dropped batch_01.json
  [2/10] dropped batch_02.json
  [3/10] dropped batch_03.json
  [4/10] dropped batch_04.json
  [5/10] dropped batch_05.json
  [6/10] dropped batch_06.json
  [7/10] dropped batch_07.json
  [8/10] dropped batch_08.json
  [9/10] dropped batch_09.json
  [10/10] dropped batch_10.json

Last progress:
{
  "id": "ac41c487-a8a4-4ee1-a8fc-7f34a8f91daf",
  "runId": "dc5bf16a-c630-4758-b4d4-166900529212",
  "name": null,
  "timestamp": "2026-05-16T21:01:50.000Z",
  "batchId": 12,
  "batchDuration": 292,
  "durationMs": {
    "triggerExecution": 292,
    "queryPlanning": 14,
    "getBatch": 0,
    "commitOffsets": 20,
    "latestOffset": 4,
    "addBatch": 226,
    "walCommit": 24
  },
  "eventTime": {
    "watermark": "2026-05-09T12:44:20.000Z"
  },
  "stateOperators": [
    {
      "operatorName": "stateStoreSave",
      "numRowsTotal": 8,
      "numRowsUpdated": 0,
      "numRowsRemoved": 10,
      "allUpdatesTimeMs"

26/05/16 23:02:00 WARN DAGScheduler: Failed to cancel job group dc5bf16a-c630-4758-b4d4-166900529212. Cannot find active jobs for it.
26/05/16 23:02:00 WARN DAGScheduler: Failed to cancel job group dc5bf16a-c630-4758-b4d4-166900529212. Cannot find active jobs for it.


## 5. Text Pipeline — Corpus → Inverted Index → Query latency → Footprint

In [7]:
corpus_schema = T.StructType([
    T.StructField("doc_id", T.StringType(), False),
    T.StructField("text",   T.StringType(), True),
])

t0 = time.time()
corpus = spark.read.schema(corpus_schema).option("header", "true").csv(CFG["paths"]["text_corpus"])
n_docs = corpus.count()
print(f"Documents: {n_docs}")

stop_words = set(CFG["text"]["stop_words"])
df_clean = corpus.withColumn("text_clean",
    F.regexp_replace(F.lower(F.col("text")), r"[^a-z0-9\s]", ""))
df_tokens = df_clean.withColumn("tokens", F.split("text_clean", r"\s+"))
df_exploded = (df_tokens.select("doc_id", F.explode("tokens").alias("token"))
               .filter(F.length("token") > 0))
tokens_before = df_exploded.count()
df_filtered = df_exploded.filter(~F.col("token").isin(stop_words))
tokens_after = df_filtered.count()

inverted_index = (df_filtered.groupBy("token")
    .agg(F.collect_list("doc_id").alias("doc_ids"),
         F.count("*").alias("freq"))
    .orderBy(F.desc("freq")))

save_plan(inverted_index, f"{CFG['paths']['proof']}/plan_index_build.txt")

idx_parquet = "outputs/project/text/parquet"
idx_csv = "outputs/project/text/csv"
for p in [idx_parquet, idx_csv]:
    if pathlib.Path(p).exists():
        shutil.rmtree(p)
pathlib.Path(idx_parquet).mkdir(parents=True, exist_ok=True)
pathlib.Path(idx_csv).mkdir(parents=True, exist_ok=True)

inverted_index.write.mode("overwrite").parquet(idx_parquet)
inverted_index.withColumn("doc_ids", F.concat_ws(",", "doc_ids")) \
    .write.mode("overwrite").option("header", "true").csv(idx_csv)

n_terms = spark.read.parquet(idx_parquet).count()
elapsed_build = (time.time() - t0) * 1000
print(f"Tokens before/after stop-words: {tokens_before} / {tokens_after}")
print(f"Unique terms: {n_terms}  •  built in {elapsed_build:.0f} ms")

log_metric("text", "index_build", "docs", n_docs, "")
log_metric("text", "index_build", "tokens_before", tokens_before, "raw tokens after explode")
log_metric("text", "index_build", "tokens_after", tokens_after, "after stop-word filter")
log_metric("text", "index_build", "unique_terms", n_terms, "")
log_metric("text", "index_build", "elapsed_ms", round(elapsed_build, 1), "")

Documents: 150


{"ts": "2026-05-16 23:03:24.734", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'for' of the type \"STRING\" cannot be cast to \"BOOLEAN\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 18 in cell [8]", "line": "", "fragment": "isin", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o346.count.\n: org.apache.spark.SparkRuntimeException: [CAST_INVALID_INPUT] The value 'for' of the type \"STRING\" cannot be cast to \"BOOLEAN\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"isin\" was called from\nline 18 in cell [8]\n\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.in

SparkRuntimeException: [CAST_INVALID_INPUT] The value 'for' of the type "STRING" cannot be cast to "BOOLEAN" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"isin" was called from
line 18 in cell [8]


In [ ]:
# Query latency benchmark — cached Parquet read
idx = spark.read.parquet(idx_parquet)
idx.cache()
idx.count()  # force cache

save_plan(idx.filter(F.col("token") == "hero"), f"{CFG['paths']['proof']}/plan_query.txt")

for term in CFG["text"]["query_terms"]:
    t0 = time.time()
    result = idx.filter(F.col("token") == term).collect()
    lat = (time.time() - t0) * 1000
    if result:
        freq = result[0]['freq']
        npost = len(result[0]['doc_ids'])
        print(f"'{term}': df={freq}, postings={npost}, latency={lat:.1f} ms")
        log_metric("text", f"query_{term}", "latency_ms", round(lat, 2), f"freq={freq}, postings={npost}")
    else:
        print(f"'{term}': NOT FOUND, latency={lat:.1f} ms")
        log_metric("text", f"query_{term}", "latency_ms", round(lat, 2), "NOT FOUND")

# Footprint comparison
p_bytes = dir_size(idx_parquet)
c_bytes = dir_size(idx_csv)
ratio = (p_bytes / c_bytes) if c_bytes else 0
print(f"\nParquet: {p_bytes:,} B  •  CSV: {c_bytes:,} B  •  ratio P/C: {ratio:.2%}")
log_metric("text", "footprint", "parquet_bytes", p_bytes, "")
log_metric("text", "footprint", "csv_bytes", c_bytes, "")
log_metric("text", "footprint", "ratio_parquet_over_csv_pct", round(ratio * 100, 2), "smaller is better")

## 6. Iterative Workload — Path A : Graph PageRank with partitioning experiment

In [8]:
# Build co-pick graph from silver: edges between heroes that played the same match
df_sv = spark.read.parquet(silver).select("match_id", "hero").distinct()

p1 = df_sv.alias("p1")
p2 = df_sv.alias("p2")
edges = (p1.join(p2, (F.col("p1.match_id") == F.col("p2.match_id")) & (F.col("p1.hero") < F.col("p2.hero")))
         .select(F.col("p1.hero").alias("src"), F.col("p2.hero").alias("dst"))
         .distinct())
edges.cache(); n_edges = edges.count()

vertices = (edges.select(F.col("src").alias("id"))
            .union(edges.select(F.col("dst").alias("id")))
            .distinct())
vertices.cache(); n_vertices = vertices.count()

# Skew analysis
all_ep = edges.select(F.col("src").alias("id")).union(edges.select(F.col("dst").alias("id")))
degree = all_ep.groupBy("id").count().withColumnRenamed("count", "degree")
print(f"Vertices: {n_vertices}  •  Edges: {n_edges}")
print("Top 5 hot vertices:")
degree.orderBy(F.desc("degree")).show(5)
degree.describe("degree").show()

log_metric("iterative", "graph_build", "vertices", n_vertices, "")
log_metric("iterative", "graph_build", "edges", n_edges, "unique co-pick pairs")

Vertices: 50  •  Edges: 1223
Top 5 hot vertices:
+-------+------+
|     id|degree|
+-------+------+
|hero_24|    49|
|hero_35|    49|
|hero_03|    49|
|hero_12|    49|
|hero_01|    49|
+-------+------+
only showing top 5 rows
+-------+------------------+
|summary|            degree|
+-------+------------------+
|  count|                50|
|   mean|             48.92|
| stddev|0.2740475156178691|
|    min|                48|
|    max|                49|
+-------+------------------+



In [9]:
# Run PageRank twice: default partitioning vs repartitioned by src
NUM_ITER = CFG["iterative"]["num_iterations"]
DAMPING = CFG["iterative"]["damping"]
N_PART = CFG["iterative"]["partitions"]

def run_pagerank(edges_in, ranks_in, out_deg, label):
    metrics = []
    prev_ranks = ranks_in
    for i in range(NUM_ITER):
        t0 = time.time()
        contribs = (edges_in.join(prev_ranks, edges_in.src == prev_ranks.id)
            .join(out_deg, "src")
            .select(F.col("dst").alias("id"),
                    (F.col("rank") / F.col("out_degree")).alias("contrib")))
        new_ranks = (contribs.groupBy("id").agg(F.sum("contrib").alias("s"))
            .withColumn("rank", F.lit((1 - DAMPING) / n_vertices) + F.lit(DAMPING) * F.col("s"))
            .select("id", "rank"))
        new_ranks = new_ranks.checkpoint()
        new_ranks.count()
        elapsed_ms = (time.time() - t0) * 1000
        delta_row = (new_ranks.alias("n").join(prev_ranks.alias("p"), "id")
                     .agg(F.max(F.abs(F.col("n.rank") - F.col("p.rank"))).alias("d")).first())
        max_delta = float(delta_row["d"]) if delta_row else 0.0
        metrics.append({"iter": i, "elapsed_ms": elapsed_ms, "max_delta": max_delta})
        print(f"[{label}] iter {i}: {elapsed_ms:.0f} ms, max_delta={max_delta:.6f}")
        prev_ranks = new_ranks
    return prev_ranks, metrics

# Run 1: default
edges_u = (edges.select(F.col("src"), F.col("dst"))
           .union(edges.select(F.col("dst").alias("src"), F.col("src").alias("dst"))))
edges_u.cache(); edges_u.count()
ranks0 = vertices.select("id").withColumn("rank", F.lit(1.0 / n_vertices))
ranks0.cache(); ranks0.count()
out_deg = edges_u.groupBy("src").count().withColumnRenamed("count", "out_degree")
out_deg.cache(); out_deg.count()

save_plan(edges_u.join(ranks0, edges_u.src == ranks0.id),
          f"{CFG['paths']['proof']}/plan_iterative_before.txt")
ranks_default, m1 = run_pagerank(edges_u, ranks0, out_deg, "default")
total1 = sum(m['elapsed_ms'] for m in m1)
print(f"\nRun 1 default total: {total1:.0f} ms")

# Run 2: repartitioned
edges_r = edges_u.repartition(N_PART, "src")
edges_r.cache(); edges_r.count()
out_deg_r = edges_r.groupBy("src").count().withColumnRenamed("count", "out_degree")
out_deg_r.cache(); out_deg_r.count()
ranks0_r = vertices.select("id").repartition(N_PART, "id").withColumn("rank", F.lit(1.0 / n_vertices))
ranks0_r.cache(); ranks0_r.count()

save_plan(edges_r.join(ranks0_r, edges_r.src == ranks0_r.id),
          f"{CFG['paths']['proof']}/plan_iterative_after.txt")
ranks_repart, m2 = run_pagerank(edges_r, ranks0_r, out_deg_r, "repart")
total2 = sum(m['elapsed_ms'] for m in m2)
print(f"Run 2 repart total: {total2:.0f} ms")
print(f"Speedup: {total1/total2:.2f}x" if total2 > 0 else "")

# Save ranks + log metrics
pathlib.Path("outputs/project/iterative").mkdir(parents=True, exist_ok=True)
ranks_default.write.mode("overwrite").parquet("outputs/project/iterative/ranks_default")
ranks_repart.write.mode("overwrite").parquet("outputs/project/iterative/ranks_repart")

for m in m1:
    log_metric("iterative", f"pagerank_default_iter_{m['iter']}", "elapsed_ms", round(m['elapsed_ms'], 1), f"max_delta={m['max_delta']:.6f}")
for m in m2:
    log_metric("iterative", f"pagerank_repart_iter_{m['iter']}",  "elapsed_ms", round(m['elapsed_ms'], 1), f"max_delta={m['max_delta']:.6f}")
log_metric("iterative", "summary", "total_default_ms", round(total1, 1), f"{NUM_ITER} iterations")
log_metric("iterative", "summary", "total_repart_ms",  round(total2, 1), f"{NUM_ITER} iterations, {N_PART} partitions")
log_metric("iterative", "summary", "speedup", round(total1/total2, 3) if total2 else 0, "default/repart")

Plan saved: proof/plan_iterative_before.txt
[default] iter 0: 455 ms, max_delta=0.000332
[default] iter 1: 302 ms, max_delta=0.000011
[default] iter 2: 284 ms, max_delta=0.000000
[default] iter 3: 308 ms, max_delta=0.000000
[default] iter 4: 274 ms, max_delta=0.000000
[default] iter 5: 256 ms, max_delta=0.000000
[default] iter 6: 258 ms, max_delta=0.000000
[default] iter 7: 256 ms, max_delta=0.000000
[default] iter 8: 198 ms, max_delta=0.000000
[default] iter 9: 203 ms, max_delta=0.000000

Run 1 default total: 2793 ms
Plan saved: proof/plan_iterative_after.txt
[repart] iter 0: 276 ms, max_delta=0.000332
[repart] iter 1: 180 ms, max_delta=0.000011
[repart] iter 2: 202 ms, max_delta=0.000000
[repart] iter 3: 199 ms, max_delta=0.000000
[repart] iter 4: 245 ms, max_delta=0.000000
[repart] iter 5: 204 ms, max_delta=0.000000
[repart] iter 6: 214 ms, max_delta=0.000000
[repart] iter 7: 182 ms, max_delta=0.000000
[repart] iter 8: 190 ms, max_delta=0.000000
[repart] iter 9: 210 ms, max_delta=0.

## 7. LLM Data Readiness — curated corpus with quality filters

In [10]:
llm_rules = CFG["llm"]

# Source 1: text corpus (hero lore, patch notes, items, match commentary)
src_corpus = spark.read.schema(corpus_schema).option("header", "true").csv(CFG["paths"]["text_corpus"])
# Source 2: commentary_text from silver (1 row per (match,hero) so deduplicate per text)
src_silver = (spark.read.parquet(silver)
              .select(F.concat(F.lit("comm_"), F.col("match_id"), F.lit("_"), F.col("hero")).alias("doc_id"),
                      F.col("commentary_text").alias("text")))

df_llm_raw = src_corpus.unionByName(src_silver)
n_llm_raw = df_llm_raw.count()

df_llm = (df_llm_raw
    .filter(F.col("text").isNotNull())
    .filter(F.length("text") >= llm_rules["min_text_length"])
    .withColumn("content_hash", F.xxhash64("text"))
    .dropDuplicates(["content_hash"])
    .withColumn("source",     F.lit(llm_rules["source"]))
    .withColumn("version",    F.lit(CFG["version"]))
    .withColumn("track",      F.lit(CFG["track"]))
    .withColumn("curated_at", F.current_timestamp())
    .withColumn("text_len",   F.length("text")))

save_plan(df_llm, f"{CFG['paths']['proof']}/plan_llm_curation.txt")

out_llm = "outputs/project/llm_ready"
if pathlib.Path(out_llm).exists():
    shutil.rmtree(out_llm)
df_llm.write.mode("overwrite").parquet(out_llm)

n_llm = spark.read.parquet(out_llm).count()
pass_rate = n_llm / n_llm_raw if n_llm_raw else 0
print(f"LLM-ready records: {n_llm:,} / {n_llm_raw:,}  ({pass_rate:.1%} pass rate)")
df_llm.printSchema()
df_llm.select("doc_id", "text_len", "source", "version").show(5, truncate=80)

log_metric("llm_prep", "curation", "raw_records", n_llm_raw, "corpus + silver commentary")
log_metric("llm_prep", "curation", "curated_records", n_llm, "after filters + dedup")
log_metric("llm_prep", "curation", "pass_rate_pct", round(pass_rate * 100, 2), "quality pass")
log_metric("llm_prep", "curation", "min_text_length", llm_rules["min_text_length"], "filter threshold")
log_metric("llm_prep", "curation", "output_bytes", dir_size(out_llm), "Parquet llm_ready footprint")

Plan saved: proof/plan_llm_curation.txt
LLM-ready records: 30,145 / 30,150  (100.0% pass rate)
root
 |-- doc_id: string (nullable = true)
 |-- text: string (nullable = true)
 |-- content_hash: long (nullable = false)
 |-- source: string (nullable = false)
 |-- version: string (nullable = false)
 |-- track: string (nullable = false)
 |-- curated_at: timestamp (nullable = false)
 |-- text_len: integer (nullable = true)

+-----------------+--------+--------------------+-------+
|           doc_id|text_len|              source|version|
+-----------------+--------+--------------------+-------+
|comm_4882_hero_02|     271|synthetic_esports_v1|   v1.0|
|comm_1580_hero_07|     270|synthetic_esports_v1|   v1.0|
|comm_2129_hero_02|     268|synthetic_esports_v1|   v1.0|
|comm_2313_hero_02|     264|synthetic_esports_v1|   v1.0|
|comm_4687_hero_08|     266|synthetic_esports_v1|   v1.0|
+-----------------+--------+--------------------+-------+
only showing top 5 rows


## 8. Evidence — write `project_metrics_log.csv`

In [11]:
metrics_path = CFG["paths"]["metrics_log"]
fieldnames = ["run_id", "stage", "task", "metric_name", "metric_value", "notes", "timestamp"]
with open(metrics_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for m in METRICS:
        w.writerow(m)

print(f"Wrote {len(METRICS)} metrics rows to {metrics_path}")
for m in METRICS:
    print(f"  [{m['stage']:9s}] {m['task']:35s} {m['metric_name']:30s} = {m['metric_value']}")

Wrote 46 metrics rows to project_metrics_log.csv
  [etl      ] bronze_land                         rows                           = 30000
  [etl      ] bronze_land                         bytes                          = 10073481
  [etl      ] bronze_land                         elapsed_ms                     = 5638.3
  [etl      ] silver_clean                        rows                           = 30000
  [etl      ] silver_clean                        bytes                          = 1598682
  [etl      ] silver_clean                        elapsed_ms                     = 3156.4
  [etl      ] silver_clean                        rows_dropped                   = 0
  [etl      ] gold_build                          team_perf_rows                 = 360
  [etl      ] gold_build                          hero_picks_rows                = 250
  [etl      ] gold_build                          bytes                          = 138272
  [etl      ] gold_build                          elapsed_ms 

## 9. SLO Check — compare against thresholds in config

In [12]:
slos = CFG["slos"]

# parquet vs csv ratio (text pipeline)
text_ratio_metric = next((m for m in METRICS if m["stage"] == "text" and m["metric_name"] == "ratio_parquet_over_csv_pct"), None)
text_ratio = (text_ratio_metric["metric_value"] / 100) if text_ratio_metric else None

llm_pass_metric = next((m for m in METRICS if m["stage"] == "llm_prep" and m["metric_name"] == "pass_rate_pct"), None)
llm_pass = (llm_pass_metric["metric_value"] / 100) if llm_pass_metric else None

checks = [
    ("parquet_csv_ratio", text_ratio, slos["parquet_csv_ratio_max"], "<="),
    ("llm_quality_pass_rate", llm_pass, slos["llm_quality_pass_rate_min"], ">="),
]

print("SLO check")
for name, val, thr, op in checks:
    if val is None:
        status = "N/A"
    elif op == "<=":
        status = "PASS" if val <= thr else "FAIL"
    else:
        status = "PASS" if val >= thr else "FAIL"
    print(f"  {name:30s}  measured={val}  threshold {op} {thr}  →  {status}")
    log_metric("slo_check", name, "status", status, f"measured={val}, threshold {op} {thr}")

# Re-write CSV including SLO checks
with open(metrics_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for m in METRICS:
        w.writerow(m)
print("\nFinal metrics log rows:", len(METRICS))

SLO check
  parquet_csv_ratio               measured=None  threshold <= 0.6  →  N/A
  llm_quality_pass_rate           measured=0.9998  threshold >= 0.8  →  PASS

Final metrics log rows: 48


In [ ]:
spark.stop()
print("Spark session stopped. Project pipeline complete.")